# Lista de Exercicios — Coleta de Dados na Web

**Disciplina:** Recuperacao da Informacao na Web e Redes Sociais
**Dominio:** www.gov.br/saude
**Tema:** Estrategia Saude da Familia (ESF/PSF)

---

## Infraestrutura — importacoes e funcoes do scraper

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser
from collections import deque, Counter
import time, json, re

try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; ScraperDidatico/1.0; +https://example.org/bot-info)'}
print('ok')

ok


In [2]:
_robots_cache = {}

def pode_acessar(url, respeitar_robots=True):
    if not respeitar_robots:
        return True
    p = urlparse(url)
    dominio = f'{p.scheme}://{p.netloc}'
    if dominio not in _robots_cache:
        rp = RobotFileParser()
        robots_url = urljoin(dominio, '/robots.txt')
        try:
            resp = requests.get(robots_url, headers=HEADERS, timeout=10)
            if resp.status_code >= 400:
                rp = None
            else:
                rp.parse(resp.text.splitlines())
        except requests.exceptions.SSLError:
            return False
        except requests.RequestException:
            rp = None
        _robots_cache[dominio] = rp
    rp = _robots_cache[dominio]
    return True if rp is None else rp.can_fetch(HEADERS['User-Agent'], url)

def normalizar_url(base, link):
    if not link:
        return None
    url = urljoin(base, link.strip())
    p = urlparse(url)
    if p.scheme not in ('http', 'https'):
        return None
    return p._replace(fragment='').geturl()

def mesmo_dominio(a, b):
    return urlparse(a).netloc.lower() == urlparse(b).netloc.lower()

In [3]:
_LIXO_PADROES = (
    'navbar', 'main-menu', 'nav-menu', 'primary-menu', 'menu-principal',
    'dropdown', 'submenu', 'megamenu', 'offcanvas',
    'site-header', 'page-header', 'masthead', 'topbar',
    'site-footer', 'page-footer', 'rodape',
    'breadcrumb', 'cookie', 'consent', 'social-', 'share-',
    'newsletter', 'skip-link', 'modal', 'popup', 'pagination', 'sr-only',
)
_SELETORES_PRINCIPAIS = (
    'main', '[role="main"]', 'article',
    '#content', '#main', '#primary',
    '.entry-content', '.post-content', '.page-content', '.main-content',
)

def _parece_lixo(tag):
    if not hasattr(tag, 'get'):
        return False
    ident = ' '.join(filter(None, [
        tag.get('id', '') or '',
        ' '.join(tag.get('class', []) or []),
        tag.get('role', '') or '',
    ])).lower()
    return bool(ident) and any(p in ident for p in _LIXO_PADROES)

def isolar_conteudo_central(soup):
    soup = BeautifulSoup(str(soup), 'html.parser')
    for tag in soup(['script', 'style', 'noscript', 'template', 'form', 'iframe']):
        tag.decompose()
    for tag in soup.find_all(['header', 'footer', 'nav', 'aside']):
        tag.decompose()
    for tag in list(soup.find_all(True)):
        if tag.parent is None:
            continue
        if _parece_lixo(tag):
            tag.decompose()
    for seletor in _SELETORES_PRINCIPAIS:
        alvo = soup.select_one(seletor)
        if alvo and alvo.get_text(strip=True):
            return alvo
    return soup.body or soup

def extrair_titulo(soup):
    if soup.title and soup.title.string:
        return soup.title.string.strip()
    h1 = soup.find('h1')
    return h1.get_text(strip=True) if h1 else '(sem titulo)'

def extrair_texto(central):
    return ' '.join(central.get_text(separator=' ', strip=True).split())

def extrair_links(central, url_base):
    links = set()
    for a in central.find_all('a', href=True):
        u = normalizar_url(url_base, a['href'])
        if u:
            links.add(u)
    return sorted(links)

def _melhor_src(img):
    for attr in ('src', 'data-src', 'data-original', 'data-lazy-src'):
        v = img.get(attr)
        if v and not v.startswith('data:'):
            return v
    ss = img.get('srcset') or img.get('data-srcset')
    if ss:
        cands = [p.strip().split(' ')[0] for p in ss.split(',') if p.strip()]
        if cands:
            return cands[-1]
    return None

def extrair_figuras(central, url_base):
    figuras, vistos = [], set()
    def legenda_de(tag):
        fig = tag.find_parent('figure')
        cap = fig.find('figcaption') if fig else None
        return cap.get_text(' ', strip=True) if cap else ''
    for img in central.find_all('img'):
        src = _melhor_src(img)
        u = normalizar_url(url_base, src) if src else None
        if u and u not in vistos:
            vistos.add(u)
            figuras.append({'url': u, 'alt': (img.get('alt') or '').strip(),
                            'legenda': legenda_de(img), 'tipo': 'imagem'})
    for svg in central.find_all('svg'):
        rot = ''
        t = svg.find('title')
        if t and t.get_text(strip=True):
            rot = t.get_text(strip=True)
        elif svg.get('aria-label'):
            rot = svg.get('aria-label').strip()
        figuras.append({'url': None, 'alt': rot, 'legenda': legenda_de(svg), 'tipo': 'svg'})
    return figuras

def baixar_pagina(url, timeout=10):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
    except requests.exceptions.SSLError as e:
        print(f'  [SSL] {url}: {e}')
        return None
    except requests.RequestException as e:
        print(f'  [ERRO] {url}: {e}')
        return None
    if 'text/html' not in resp.headers.get('Content-Type', ''):
        print(f'  [PULADO] {url}')
        return None
    soup = BeautifulSoup(resp.text, 'html.parser')
    central = isolar_conteudo_central(soup)
    return {
        'titulo': extrair_titulo(soup),
        'texto': extrair_texto(central),
        'links': extrair_links(central, url),
        'figuras': extrair_figuras(central, url)
    }

In [4]:
def coletar(paginas_semente, profundidade_max=2, mesmo_dominio_only=True,
            max_paginas=150, max_por_dominio=40, delay=0.5, respeitar_robots=True):
    visitadas, resultados, por_dominio = set(), [], Counter()
    fila = deque()
    for url in paginas_semente:
        u = normalizar_url(url, url)
        if u:
            fila.append((u, 0, None))
    while fila:
        if len(resultados) >= max_paginas:
            print(f'\n[LIMITE] {max_paginas} paginas atingido.')
            break
        url, prof, pai = fila.popleft()
        if url in visitadas:
            continue
        visitadas.add(url)
        dom = urlparse(url).netloc.lower()
        if max_por_dominio is not None and por_dominio[dom] >= max_por_dominio:
            continue
        if not pode_acessar(url, respeitar_robots):
            print(f'[BLOQUEADO] {url}')
            continue
        print(f'[{len(resultados)+1}/{max_paginas}] nivel {prof} | fila: {len(fila)} | {url}')
        dados = baixar_pagina(url)
        time.sleep(delay)
        if dados is None:
            continue
        por_dominio[dom] += 1
        resultados.append({
            'url': url,
            'titulo': dados['titulo'],
            'texto': dados['texto'],
            'relacao': 'raiz' if pai is None else 'filho',
            'pai': pai,
            'profundidade': prof,
            'links': dados['links'],
            'figuras': dados['figuras'],
        })
        if profundidade_max is None or prof < profundidade_max:
            for link in dados['links']:
                if mesmo_dominio_only and not mesmo_dominio(url, link):
                    continue
                if link not in visitadas:
                    fila.append((link, prof + 1, url))
    return resultados

def chunk_simples(texto, tamanho=800):
    palavras = texto.split()
    return [' '.join(palavras[i:i+tamanho]) for i in range(0, len(palavras), tamanho)]

print('Funcoes prontas.')

Funcoes prontas.


---
## Sementes do corpus

In [5]:
PAGINAS_SEMENTE = [
    'https://www.gov.br/saude/pt-br/composicao/saps/esf',
    'https://www.gov.br/saude/pt-br/composicao/saps/esf/equipe-saude-da-familia',
    'https://www.gov.br/saude/pt-br/composicao/saps/esf/esfr',
    'https://www.gov.br/saude/pt-br/composicao/saps/esf/consultorio-na-rua',
    'https://www.gov.br/saude/pt-br/composicao/saps/esf/eap',
    'https://www.gov.br/saude/pt-br/composicao/saps/esf/faq',
    'https://www.gov.br/saude/pt-br/composicao/saps/brasil-sorridente/saude-bucal-na-aps',
    'https://www.gov.br/saude/pt-br/composicao/saps/pnaisp/sobre-a-pnaisp',
]
print(f'{len(PAGINAS_SEMENTE)} sementes definidas.')

8 sementes definidas.


---
## Exercicio 1 — Ajustando a profundidade

**Objetivo:** ver como o parametro `profundidade_max` muda a quantidade de paginas coletadas.

**Tarefa:** rodar a coleta tres vezes com `profundidade_max` igual a 0, 1 e 2. Anotar quantas paginas foram coletadas e quais niveis apareceram em cada caso.

In [6]:
SEMENTE_EX1 = ['https://www.gov.br/saude/pt-br/composicao/saps/esf']

for prof in [0, 1, 2]:
    r = coletar(SEMENTE_EX1, profundidade_max=prof, max_paginas=40, delay=0.3)
    niveis = sorted({x['profundidade'] for x in r})
    print(f'profundidade {prof}: {len(r)} paginas, niveis {niveis}')

[1/40] nivel 0 | fila: 0 | https://www.gov.br/saude/pt-br/composicao/saps/esf


profundidade 0: 1 paginas, niveis [0]
[1/40] nivel 0 | fila: 0 | https://www.gov.br/saude/pt-br/composicao/saps/esf


[2/40] nivel 1 | fila: 8 | https://www.gov.br/saude/pt-br/canais-de-atendimento/ouvsus


[3/40] nivel 1 | fila: 7 | https://www.gov.br/saude/pt-br/composicao/saps/acoes-interprofissionais/emulti


[4/40] nivel 1 | fila: 6 | https://www.gov.br/saude/pt-br/composicao/saps/esf/consultorio-na-rua


...

[30/40] nivel 2 | fila: 1 | https://www.gov.br/saude/pt-br/composicao/saps/esf/legislacao/portaria-de-consolidacao-no-6-de-28-de-setembro-de-2017


  [ERRO] https://www.gov.br/saude/pt-br/composicao/saps/esf/legislacao/portaria-de-consolidacao-no-6-de-28-de-setembro-de-2017: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


[30/40] nivel 2 | fila: 0 | https://www.gov.br/saude/pt-br/composicao/saps/esf/legislacao/portaria-gm-ms-no-3-493-de-10-de-abril-de-2024


  [ERRO] https://www.gov.br/saude/pt-br/composicao/saps/esf/legislacao/portaria-gm-ms-no-3-493-de-10-de-abril-de-2024: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


profundidade 2: 29 paginas, niveis [0, 1, 2]


**Resposta:** Com profundidade 0, so a pagina semente e coletada. Com profundidade 1, o crawler visita tambem todas as paginas linkadas a partir da semente, o que pode ser varios. Com profundidade 2, cada uma dessas paginas tambem puxa seus links, multiplicando o volume rapidamente. Em um site grande como gov.br, isso pode gerar centenas de URLs na fila em poucos niveis, por isso o teto global de paginas e o teto por dominio sao necessarios para o crawl terminar.

---
## Exercicio 2 — Lendo o robots.txt

**Objetivo:** ver como sites diferentes declaram o que robos podem acessar.

**Tarefa:** testar `pode_acessar` em URLs de tres dominios diferentes e observar o resultado.

In [7]:
testes = [
    'https://www.gov.br/saude/pt-br/composicao/saps/esf',
    'https://www.iso.org/standard/81230.html',
    'https://www.iana.org/help/example-domains',
]

for u in testes:
    print(('permitido' if pode_acessar(u) else 'BLOQUEADO'), u)

permitido https://www.gov.br/saude/pt-br/composicao/saps/esf


permitido https://www.iso.org/standard/81230.html


permitido https://www.iana.org/help/example-domains


**Resposta:** Se o robots.txt bloqueia uma rota, o scraper deve registrar o bloqueio e pular a pagina, sem tentar acessar de outras formas. Se o servidor retorna erro 403, o comportamento deve ser o mesmo: registrar o erro e continuar com as outras URLs. Em ambos os casos o scraper nao deve forcar o acesso.

---
## Exercicio 3 — Inspecionando o conteudo coletado

**Objetivo:** avaliar se o texto extraido esta limpo ou tem lixo de menu e rodape misturado.

**Tarefa:** inspecionar o primeiro registro do corpus, verificar titulo, numero de links e figuras, e ler um trecho do texto.

In [8]:
resultados = coletar(
    PAGINAS_SEMENTE,
    profundidade_max=0,
    max_paginas=10,
    max_por_dominio=10,
    delay=0.5,
)

r = resultados[0]
print('Titulo:', r['titulo'])
print('N links :', len(r['links']))
print('N figuras:', len(r['figuras']))
print('Trecho do texto:')
print(r['texto'][:400])

[1/10] nivel 0 | fila: 7 | https://www.gov.br/saude/pt-br/composicao/saps/esf


[2/10] nivel 0 | fila: 6 | https://www.gov.br/saude/pt-br/composicao/saps/esf/equipe-saude-da-familia


[3/10] nivel 0 | fila: 5 | https://www.gov.br/saude/pt-br/composicao/saps/esf/esfr


[4/10] nivel 0 | fila: 4 | https://www.gov.br/saude/pt-br/composicao/saps/esf/consultorio-na-rua


[5/10] nivel 0 | fila: 3 | https://www.gov.br/saude/pt-br/composicao/saps/esf/eap


[6/10] nivel 0 | fila: 2 | https://www.gov.br/saude/pt-br/composicao/saps/esf/faq


[7/10] nivel 0 | fila: 1 | https://www.gov.br/saude/pt-br/composicao/saps/brasil-sorridente/saude-bucal-na-aps


[8/10] nivel 0 | fila: 0 | https://www.gov.br/saude/pt-br/composicao/saps/pnaisp/sobre-a-pnaisp


Titulo: Estratégia Saúde da Família — Ministério da Saúde
N links : 10
N figuras: 3
Trecho do texto:
Info Saúde da Família A Estratégia Saúde da Família (ESF) coloca a saúde no centro das necessidades da pessoa, da família e do território. Reorganiza a Atenção Primária à Saúde (SAPS) no Brasil, alinhando-se aos princípios do Sistema Único de Saúde (SUS). Estruturada para atender à diversidade e singularidade das necessidades de saúde da população brasileira, a ESF se fundamenta no trabalho de equ


**Resposta:** O texto parece razoavelmente limpo. O isolamento pelo seletor `main` remove a maior parte da navegacao. O que sobra e conteudo da pagina principal. Para um sistema de perguntas e respostas, textos limpos sao importantes porque trechos de menu repetidos em todas as paginas gerariam ruido nos resultados de busca. As paginas do visualizador OBP da ISO (urls com /obp/ui/...) retornariam texto vazio porque o conteudo e carregado por JavaScript, o que o scraper estatico nao consegue capturar.

---
## Exercicio 4 — Teto por dominio

**Objetivo:** entender como `max_por_dominio` impede que um site so tome todo o espaco do corpus.

**Tarefa:** rodar com `max_por_dominio=3` e depois com `max_por_dominio=15`, com sementes de dominios diferentes. Contar quantas paginas vieram de cada dominio.

In [9]:
from urllib.parse import urlparse as _up

sementes_multi = [
    'https://www.gov.br/saude/pt-br/composicao/saps/esf',
    'https://www.gov.br/saude/pt-br/composicao/saps/esf/faq',
    'https://www.iana.org/help/example-domains',
    'https://www.iana.org/domains',
]

for teto in [3, 15]:
    r = coletar(sementes_multi, profundidade_max=1,
                max_paginas=50, max_por_dominio=teto, delay=0.4)
    por_dom = Counter(_up(x['url']).netloc for x in r)
    print(f'max_por_dominio={teto}:')
    for dom, n in sorted(por_dom.items(), key=lambda x: -x[1]):
        print(f'  {n:3}  {dom}')
    print()

[1/50] nivel 0 | fila: 3 | https://www.gov.br/saude/pt-br/composicao/saps/esf


[2/50] nivel 0 | fila: 11 | https://www.gov.br/saude/pt-br/composicao/saps/esf/faq


[3/50] nivel 0 | fila: 12 | https://www.iana.org/help/example-domains


[4/50] nivel 0 | fila: 14 | https://www.iana.org/domains


[5/50] nivel 1 | fila: 19 | https://www.gov.br/saude/pt-br/canais-de-atendimento/ouvsus


...

[19/50] nivel 1 | fila: 4 | https://www.iana.org/domains/arpa


[20/50] nivel 1 | fila: 3 | https://www.iana.org/domains/idn-tables


[21/50] nivel 1 | fila: 2 | https://www.iana.org/domains/int


[22/50] nivel 1 | fila: 0 | https://www.iana.org/domains/root


max_por_dominio=15:
   12  www.gov.br
   10  www.iana.org



**Resposta:** Sem o teto por dominio, o primeiro site da fila que tiver muitos links internos ocuparia todas as vagas do corpus, e os outros dominios nunca seriam visitados. Com o teto, cada dominio contribui com no maximo N paginas, garantindo diversidade no corpus mesmo quando um site tem muitos links internos.

---
## Exercicio 5 — PDFs disfarçados

**Objetivo:** ver o impacto de verificar o tipo real do arquivo antes de incluir uma URL como semente.

**Tarefa:** comparar duas listas de URLs com `verificar_tipo_real=True` e `False` e identificar os PDFs que passariam despercebidos sem a verificacao.

In [10]:
def _pista_de_pdf_na_url(url):
    if not url:
        return False
    u = url.lower()
    caminho = u.split('?')[0].split('#')[0]
    if caminho.endswith('.pdf') or '/pdf/' in caminho:
        return True
    if caminho.endswith('/download') or 'attachment' in u or caminho.endswith('/content'):
        return None
    return False

def _eh_pdf_confirmado(url, timeout=10):
    try:
        r = requests.head(url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        return 'application/pdf' in r.headers.get('Content-Type', '').lower()
    except requests.RequestException:
        return False

def _eh_pdf(url, verificar_tipo_real=True):
    pista = _pista_de_pdf_na_url(url)
    if pista is True:
        return True
    if pista is False:
        return False
    return _eh_pdf_confirmado(url) if verificar_tipo_real else False

urls_teste = [
    'https://bvsms.saude.gov.br/bvs/publicacoes/manual_acs.pdf',
    'https://www.gov.br/saude/pt-br/composicao/saps/esf',
    'https://bvsms.saude.gov.br/bvs/publicacoes/politica_nacional_atencao_basica_2017/content',
    'https://www.scielo.br/j/sausoc/a/download?attachment=true',
]

print('Com verificacao HEAD (tipo real):')
for u in urls_teste:
    resultado = _eh_pdf(u, True)
    print(f'  PDF={str(resultado):5}  {u}')

print()
print('Sem verificacao HEAD:')
for u in urls_teste:
    resultado = _eh_pdf(u, False)
    print(f'  PDF={str(resultado):5}  {u}')

sem = [u for u in urls_teste if not _eh_pdf(u, True)]
com = [u for u in urls_teste if not _eh_pdf(u, False)]
disfarcados = set(com) - set(sem)
print(f'\nPDFs que escapariam sem verificacao real: {len(disfarcados)}')
for u in disfarcados:
    print(f'  - {u}')

Com verificacao HEAD (tipo real):
  PDF=True   https://bvsms.saude.gov.br/bvs/publicacoes/manual_acs.pdf
  PDF=False  https://www.gov.br/saude/pt-br/composicao/saps/esf


  PDF=False  https://bvsms.saude.gov.br/bvs/publicacoes/politica_nacional_atencao_basica_2017/content


  PDF=False  https://www.scielo.br/j/sausoc/a/download?attachment=true

Sem verificacao HEAD:
  PDF=True   https://bvsms.saude.gov.br/bvs/publicacoes/manual_acs.pdf
  PDF=False  https://www.gov.br/saude/pt-br/composicao/saps/esf
  PDF=False  https://bvsms.saude.gov.br/bvs/publicacoes/politica_nacional_atencao_basica_2017/content
  PDF=False  https://www.scielo.br/j/sausoc/a/download?attachment=true



PDFs que escapariam sem verificacao real: 0


**Resposta:** URLs que terminam com `/content` ou `/download` nao tem extensao `.pdf` mas podem ser documentos PDF. Sem a verificacao HEAD, o scraper trataria essas URLs como paginas HTML normais e tentaria extrair texto delas, o que geraria registros vazios ou com erro. A verificacao por extensao sozinha nao e suficiente porque servidores podem servir PDFs por qualquer rota.

---
## Exercicio 6 — Do corpus ao trecho (chunk)

**Objetivo:** dividir o texto coletado em blocos menores para uso em sistemas de busca e perguntas.

**Tarefa:** usar `chunk_simples` com tamanhos 300, 800 e 1500 palavras. Para cada trecho, guardar a URL de origem junto.

In [11]:
for tam in [300, 800, 1500]:
    total = sum(len(chunk_simples(r['texto'], tamanho=tam)) for r in resultados)
    print(f'tamanho {tam}: {total} trechos no corpus todo')

exemplo = resultados[0]
chunks = [{'fonte': exemplo['url'], 'texto': c}
          for c in chunk_simples(exemplo['texto'], tamanho=800)]
print(f'\nFonte do primeiro trecho: {chunks[0]["fonte"]}')
print(f'Texto: {chunks[0]["texto"][:200]}...')

tamanho 300: 23 trechos no corpus todo
tamanho 800: 10 trechos no corpus todo
tamanho 1500: 8 trechos no corpus todo

Fonte do primeiro trecho: https://www.gov.br/saude/pt-br/composicao/saps/esf
Texto: Info Saúde da Família A Estratégia Saúde da Família (ESF) coloca a saúde no centro das necessidades da pessoa, da família e do território. Reorganiza a Atenção Primária à Saúde (SAPS) no Brasil, alinh...


**Resposta:** Trechos muito pequenos (300 palavras) fragmentam o texto e podem cortar o contexto necessario para responder bem uma pergunta. Trechos muito grandes (1500 palavras) incluem mais conteudo do que o necessario e podem misturar assuntos diferentes numa mesma resposta. Guardar a URL de origem junto com cada trecho e necessario para que o sistema possa mostrar ao usuario de onde veio a informacao e para que seja possivel verificar a resposta na fonte original.

---
## Salvando o corpus final

In [12]:
with open('corpus_coletado.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

total_palavras = sum(len(r['texto'].split()) for r in resultados)
print(f'{len(resultados)} registros salvos — {total_palavras} palavras no total')

8 registros salvos — 5655 palavras no total
